In [1]:
import os
import pandas as pd
import json
from dotenv import load_dotenv 


load_dotenv()

True

In [2]:
# load cofig

with open("config.json", "r") as f:
    config = json.load(f)

# Load trancript
df = pd.read_excel("data/transcripts.xlsx")

df.head(2)

,call_id,agent_name,transcript,expected_call_type
0,1,Matt,Customer: I was charged twice for my premium t...,billing
1,2,Kim,Customer: My claim has been pending for 2 week...,claims


In [3]:
# Initializing LLM
from langchain_openai import ChatOpenAI

provider = config["provider"]
model_cnfg = config[provider]

if provider == "ollama":
    llm = ChatOpenAI(
        model = model_cnfg["model"],
        temperature= model_cnfg["temperature"],
        max_tokens = model_cnfg["max_tokens"],
        base_url= "http://localhost:11434/v1"
    )
else:
    raise ValueError(f"Invalid provider '{provider}' in config.json")

print(f"LLM config sucessfull {provider}, model {model_cnfg["model"]} ✅")


LLM config sucessfull ollama, model qwen2.5:7b-instruct ✅


In [4]:
response = llm.invoke("in one line say that I am able to reach you\n")
print(response.content)

You can reach me at any time through this chat interface.


BaseModel lets you create that form.
` from pydantic import BaseModel`
`class Student(BaseModel):`
    `name: str`
    `age: int`
    `grade: str` 

Read this like:

"I am creating a Student form.

Every student must have:

a name (text)
an age (number)
a grade (text)"

BaseModel Checks Data and if the data does not match the category it' define give an error. 

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# define output schema
class ClassificationOutput(BaseModel):
    call_type: str = Field(description= "Type of customer call")
    confidence: str = Field(description= "Confidence score between 0 and 1")

parser = PydanticOutputParser(pydantic_object=ClassificationOutput) # need to an pydantic object which is the class we created. 


# prompt templete
prompt = PromptTemplate(
    template="""
You are a call classification assistant.
Classify the following customer support transcript into one of these categories:
{labels}

Transcript:
{transcript}

{format_instructions}
""",
    input_variables= ["transcript"],
    partial_variables={
        "format_instructions": parser.get_format_instructions(),
        "labels": config["classification"]["labels"]
    }
)

# prompt chain

classification_chain = prompt | llm | parser

# test with one transcript or record

# sample_text = df.iloc[0]["transcript"]

# result = classification_chain.invoke({
#     "transcript": sample_text
# })

# print(f"Classification Result: \n{result}")

## PromptTemplate Purpose

Creates a reusable prompt for classifying customer support call transcripts.

### What it does

- Defines the prompt text sent to the LLM.
- Inserts a list of classification categories (`labels`).
- Inserts the customer call transcript (`transcript`).
- Inserts output formatting instructions from a parser (`format_instructions`).
- Requires only the `transcript` to be provided at runtime.
- Pre-fills `labels` and `format_instructions` using `partial_variables`.

### Runtime Flow

1. Receive a transcript.
2. Replace `{transcript}` with the actual transcript.
3. Replace `{labels}` with configured classification labels.
4. Replace `{format_instructions}` with parser formatting instructions.
5. Send the completed prompt to the LLM.
6. Receive a structured classification result.


In [6]:
# loading 

from tqdm import tqdm

results = []

for i, row in tqdm(df.iterrows(), total = len(df), desc = "Classifing Calls"):
    try:
        output = classification_chain.invoke(
            {
                "transcript": row["transcript"] # row is what we are itterating 
            }
        )

        results.append(
            {
                "call_id": row["call_id"],
                "predicted_call_type": output.call_type,
                "confidence": output.confidence
            }
        )
    except Exception as e:
        print(f"Error at row {i}:{e}")
        results.append(
            {
            "call_id": row["call_id"],
            "predicted_call_type": None,
            "confidence": None
            }
        )

result_df = pd.DataFrame(results)
df = df.merge(result_df, on= "call_id")

print("Batch Classification Completed")
print(df.head(1))

Classifing Calls: 100%|██████████| 4/4 [00:16<00:00,  4.05s/it]

Batch Classification Completed
   call_id agent_name                                         transcript  \
0        1       Matt  Customer: I was charged twice for my premium t...   

  expected_call_type predicted_call_type confidence  
0            billing             billing          1  


In [7]:
df.head(2)

,call_id,agent_name,transcript,expected_call_type,predicted_call_type,confidence
0,1,Matt,Customer: I was charged twice for my premium t...,billing,billing,1
1,2,Kim,Customer: My claim has been pending for 2 week...,claims,claims,1


In [8]:
# define routing logic 

def route_call(call_type):
    if call_type  == "billing":
        return ["knowledge_accuracy", "resolution_quality"]
    if call_type  == "claims":
        return ["knowledge_accuracy", "resolution_quality"]
    if call_type  == "complaint":
        return ["tone_empathy", "resolution_quality"]
    if call_type  == "general_query":
        return ["knowledge_accuracy"]
    else:
        return ["knowledge_accuracy"] # fallback

# apply routing to our datasets

df["evaluation_plan"] = df["predicted_call_type"].apply(route_call)

In [9]:
df.head(2)

,call_id,agent_name,transcript,expected_call_type,predicted_call_type,confidence,evaluation_plan
0,1,Matt,Customer: I was charged twice for my premium t...,billing,billing,1,"[knowledge_accuracy, resolution_quality]"
1,2,Kim,Customer: My claim has been pending for 2 week...,claims,claims,1,"[knowledge_accuracy, resolution_quality]"


In [10]:
# define output schema

class ToneEvaluation(BaseModel):
    score: int = Field(description= "Score between 1 and 5 1 being good and 5 being bad")
    reasoning: str = Field(description= "Explanation of the score")

tone_parser = PydanticOutputParser(pydantic_object= ToneEvaluation)

tone_prompt = PromptTemplate(
    template= """
You are a QA evaluator for customer support calls.

Evaluate the agent"s tone and empthy in the following transcript.

Consider:
- Did the agent acknowledge the customer"s issue?
- Was the tone polite and professional?
- Did the agent show empathy?

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": tone_parser.get_format_instructions
    }
)

#define chain

tone_chain = tone_prompt | llm | tone_parser

# test with one transcript or record

sample_text = df[df["predicted_call_type"] == "complaint"].iloc[0]["transcript"]

result = tone_chain.invoke({
    "transcript": sample_text
})

print(f"Tone Evaluation Result: \n{result}")


Tone Evaluation Result: 
score=3 reasoning="The agent acknowledged the customer's issue by expressing apology but the tone seems a bit abrupt and lacks warmth. The agent could have further improved by showing more empathy and reassuring the customer that their concerns are being taken seriously."


In [11]:
# resoluation 

class ResolutionEvaluation(BaseModel):
    score: int = Field(description= "Score between 1 and 5. 1 being good and 5 being bad")
    reasoning: str = Field(description= "Explaination of the score")

resolution_parser = PydanticOutputParser(pydantic_object=ResolutionEvaluation)

# ---- Step 2 : Promt ------

resolution_promot = PromptTemplate(
    template= """
You are a QA evaluator for customer support calls.

Evaluate the resolution quality of the agent.

Consider:
- Did the agent fully resolve the customer's issue?
- Were next steps clearly communicated?
- Did the agent confirm resolution before ending?

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": resolution_parser.get_format_instructions()
    }
)

#define chain

resolution_chain = resolution_promot | llm | resolution_parser

# test with one transcript or record

sample_text = df.iloc[0]["transcript"]

result = tone_chain.invoke({
    "transcript": sample_text
})

print(f"✅ Tone Evaluation Result: \n{result}")

✅ Tone Evaluation Result: 
score=3 reasoning="The agent started to check the issue, which is a positive step. However, the tone seems neutral and could have been more polite and professional. Additionally, there was no indication of empathy towards the customer's frustration with being charged twice."


In [12]:
# knowladge chain

class KnowledgeEvaluation(BaseModel):
    score: int = Field(description= "Score between 1 and 5 1 being good and 5 being bad")
    reasoning: str = Field(description= "Explanation of the score")

knowladge_parser = PydanticOutputParser(pydantic_object= KnowledgeEvaluation)

# ----- Step 3: Promt ----
knowladge_promot = PromptTemplate(
    template= """
You are a QA evaluator for customer support calls.

Evaluate the resolution quality of the agent.

Consider:
- Did the agent provide correct and relevant information?
- Was the explanation clear and easy to understand?
- Did the agent avoid vague or misleading statments?

IMPORTANT:
- If the transcript does not contain enough information, give a moderate score (2 or  3) and explain

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": knowladge_parser.get_format_instructions()
    }
)

#define chain

knowledge_chain = knowladge_promot | llm | knowladge_parser

# test with one transcript or record

sample_text = df.iloc[0]["transcript"]

result = tone_chain.invoke({
    "transcript": sample_text
})

print(f"✅ Tone Evaluation Result: \n{result}")

✅ Tone Evaluation Result: 
score=3 reasoning="The agent acknowledges the customer's issue by saying 'Let me check that for you...', which is a positive start. However, the tone is neutral and the agent does not show any empathy towards the customer's frustration with being charged twice. A more empathetic response would have included an apology and perhaps an acknowledgment of the inconvenience caused."


In [13]:
# evaluation runner

def run_evaluations(transcript, eval_plan):
    results = {}

    if "tone_empathy" in eval_plan:
        try:
            tone_result = tone_chain.invoke({"transcript": transcript})
            results["tone"] = tone_result.model_dump()
        except Exception as e:
            results["tone"] = {"error": str(e)}

    if "knowledge_accuracy" in eval_plan:
        try:
            knowledge_result = knowledge_chain.invoke({"transcript": transcript})
            results["knowledge"] = knowledge_result.model_dump()
        except Exception as e:
            results["knowledge"] = {"error": str(e)}

    if "resolution_quality" in eval_plan:
        try:
            resolution_result = resolution_chain.invoke({"transcript": transcript})
            results["resolution"] = resolution_result.model_dump()
        except Exception as e:
            results["resolution"] = {"error": str(e)}

    return results


# --- Step 2: Apply to entire dataset ---

evaluation_outputs = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Running Evaluations"):
    output = run_evaluations(row["transcript"], row["evaluation_plan"])
    
    evaluation_outputs.append({
        "call_id": row["call_id"],
        "evaluation_output": output
    })

# Convert to DataFrame
eval_df = pd.DataFrame(evaluation_outputs)

# Merge
df = df.merge(eval_df, on="call_id")

print("\n✅ Evaluation Completed\n")

# Show one example clearly
import pprint
pprint.pprint(df.iloc[0]["evaluation_output"])

Running Evaluations: 100%|██████████| 4/4 [01:22<00:00, 20.71s/it]


✅ Evaluation Completed

{'knowledge': {'reasoning': 'The transcript does not provide enough '
                            'information from the agent to evaluate the '
                            'resolution quality. The agent initiated an action '
                            'to check the issue but did not provide any '
                            'further details or resolution steps. This lack of '
                            'information makes it difficult to assess the '
                            'correctness, clarity, or avoidance of vague '
                            'statements.',
               'score': 2},
 'resolution': {'reasoning': 'The agent started to check the issue but did not '
                             'confirm if the problem was resolved or provide '
                             "next steps. The customer's concern about being "
                             'charged twice was not fully addressed or '
                             'resolved.',
                'sc

In [15]:
class FinalReport(BaseModel):
    summary: str = Field(description="Overall evaluation summary")
    recommendations: list[str] = Field(description="List of actionable improvements")

final_parser = PydanticOutputParser(pydantic_object=FinalReport)

# --- Step 2: Prompt ---
final_prompt = PromptTemplate(
    template="""
You are a QA manager reviewing customer support calls.

Based on the evaluation results below, generate:

1. A concise summary of the agent's performance
2. A list of actionable recommendations for improvement

Evaluation Data:
{evaluation_output}

IMPORTANT:
- Be specific and practical
- Do not repeat scores
- Focus on improvement

{format_instructions}
""",
    input_variables=["evaluation_output"],
    partial_variables={
        "format_instructions": final_parser.get_format_instructions()
    }
)

# --- Step 3: Chain ---
final_chain = final_prompt | llm | final_parser

# --- Step 4: Test on one row
sample_eval = df.iloc[0]["evaluation_output"]

result = final_chain.invoke({
    "evaluation_output": sample_eval
})

print("✅ Final QA Report:")
print(result)

✅ Final QA Report:
summary="The agent demonstrated basic knowledge and initiated the process to check the issue, but the interaction lacked detail and clarity, making it impossible to assess the resolution's correctness. The customer's concern was not fully addressed, and no next steps were provided." recommendations=['Provide clear and detailed explanations of the steps being taken to resolve the issue.', 'Follow up with the customer to confirm the resolution and gather feedback.', 'Offer clear next steps if the issue is still unresolved.', 'Improve the knowledge base to ensure agents can provide more detailed and precise information.']


In [16]:
from tqdm import tqdm

final_outputs = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Generating Final Reports"):
    try:
        result = final_chain.invoke({
            "evaluation_output": row["evaluation_output"]
        })

        final_outputs.append({
            "call_id": row["call_id"],
            "summary": result.summary,
            "recommendations": result.recommendations
        })

    except Exception as e:
        print(f"❌ Error at row {i}: {e}")
        final_outputs.append({
            "call_id": row["call_id"],
            "summary": None,
            "recommendations": None
        })

# Convert to DataFrame
final_df = pd.DataFrame(final_outputs)

# Merge
df = df.merge(final_df, on="call_id")

print("\n✅ Final Reports Generated\n")

# Show final result
df[[
    "call_id",
    "predicted_call_type",
    "evaluation_output",
    "summary",
    "recommendations"
]]

Generating Final Reports: 100%|██████████| 4/4 [01:10<00:00, 17.54s/it]


✅ Final Reports Generated



,call_id,predicted_call_type,evaluation_output,summary,recommendations
0,1,billing,"{'knowledge': {'score': 2, 'reasoning': 'The t...",The agent made an initial step to check the is...,[Provide clear and detailed steps to resolve t...
1,2,claims,"{'knowledge': {'score': 2, 'reasoning': 'The a...",The agent demonstrated satisfactory resolution...,[Provide more detailed and specific informatio...
2,3,complaint,"{'tone': {'score': 3, 'reasoning': 'The agent ...",The agent demonstrated a basic level of custom...,[Enhance the empathetic tone to better connect...
3,4,general_query,"{'knowledge': {'score': 3, 'reasoning': 'The t...",The agent demonstrated a satisfactory level of...,[Request more detailed transcripts for a thoro...


In [17]:
accuracy = (df["expected_call_type"] == df["predicted_call_type"]).mean()
print(f"🎯 Classification Accuracy: {accuracy:.2f}")

🎯 Classification Accuracy: 1.00


In [18]:
df.to_excel("data/output.xlsx", index=False)
print("✅ Results saved to data/output.xlsx")

✅ Results saved to data/output.xlsx
